In [1]:
import pandas as pd
import numpy as np
import altair as alt
alt.data_transformers.enable("vegafusion")
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

In [13]:
df_transactions = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/raw/transactions.parquet', engine='pyarrow')

In [14]:
df_transactions['date'] = pd.to_datetime(df_transactions['date'])
df_transactions

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
...,...,...,...
83483,2017-08-15,50,2804
83484,2017-08-15,51,1573
83485,2017-08-15,52,2255
83486,2017-08-15,53,932


In [15]:
df_transactions[df_transactions['transactions'] < 100]

,date,store_nbr,transactions
52392,2016-01-02,2,6
52428,2016-01-04,1,10
57950,2016-04-17,53,33
58003,2016-04-18,53,54
65479,2016-09-07,43,5
74018,2017-02-20,30,97


In [17]:
# Define the date range you want to ensure is covered
start_date = '2013-01-01'
end_date = '2017-08-15'
complete_date_range = pd.date_range(start=start_date, end=end_date)

# Create a DataFrame with all combinations of store_nbr and complete_date_range
stores = df_transactions['store_nbr'].unique()

df_dates = pd.DataFrame({
    'date': pd.concat([pd.Series(complete_date_range)] * len(stores), ignore_index=True),
    'store_nbr': sorted(list(stores) * len(complete_date_range))
})

df_dates['date'] = pd.to_datetime(df_dates['date'])

df_dates

,date,store_nbr
0,2013-01-01,1
1,2013-01-02,1
2,2013-01-03,1
3,2013-01-04,1
4,2013-01-05,1
...,...,...
91147,2017-08-11,54
91148,2017-08-12,54
91149,2017-08-13,54
91150,2017-08-14,54


In [20]:
df_merged = pd.merge(df_dates, df_transactions, on=['store_nbr', 'date'], how='left')

df_merged.fillna({'transactions': 0}, inplace = True)

df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91152 entries, 0 to 91151
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          91152 non-null  datetime64[ns]
 1   store_nbr     91152 non-null  int64         
 2   transactions  91152 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 2.1 MB


In [25]:
# Create a flag column for coloring
df_merged['color_flag'] = df_merged['transactions'].apply(lambda x: 'colored' if x > 0 else 'uncolored')

chart = alt.Chart(df_merged).mark_bar().encode(
    x='date:T',
    y='store_nbr:N',
    color=alt.condition(
        alt.datum.transactions > 0,  # condition for coloring
        'store_nbr:N',               # color based on store number for transactions > 0
        alt.value('black')        # no color for transactions == 0
    ),
    tooltip=['date', 'store_nbr', 'transactions']
).properties(
    width=1000,
    height=600,
    title="Transaction Timeline per Store"
)

chart.display()

alt.Chart(...)